# Logistic Regression - Fixed Evaluation

This notebook evaluates Logistic Regression with a temporal train/validation/test split.

Main fixes:
- no Google Colab paths;
- validation set is used for threshold selection;
- validation set is used for threshold selection, then the final model is refit on train+validation;
- test set is used only once for final reporting;
- PR-AUC, F1, recall, precision and Balanced Accuracy are reported for both scenarios;
- SMOTE is placed inside an `imblearn.Pipeline`, so resampling is fit only on training data.


In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42


In [2]:
def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Models" / "processed_bank_data.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate Models/processed_bank_data.csv")

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "Models" / "processed_bank_data.csv"

df = pd.read_csv(DATA_PATH)
print("Data path:", DATA_PATH)
print("Shape:", df.shape)
print("Positive rate:", f"{df['y'].mean():.2%}")
display(df.head())


Data path: H:\Nam ba\Hoc ki 2\May Hoc\Project ML\Project after fixed\Models\processed_bank_data.csv
Shape: (41176, 58)
Positive rate: 11.27%


,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,...,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success,age_group_adult,age_group_middle,age_group_senior,y
0,56,261,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,True,False,0
1,57,149,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,True,False,0
2,37,226,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,True,False,False,0
3,40,151,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,True,False,False,0
4,56,307,1,999,0,1.1,93.994,-36.4,4.857,5191.0,...,True,False,False,False,True,False,False,True,False,0


In [3]:
def temporal_split(frame: pd.DataFrame, target: str = "y", train_size: float = 0.7, valid_size: float = 0.1):
    n = len(frame)
    train_end = int(n * train_size)
    valid_end = int(n * (train_size + valid_size))

    train = frame.iloc[:train_end].copy()
    valid = frame.iloc[train_end:valid_end].copy()
    test = frame.iloc[valid_end:].copy()

    splits = {}
    for name, part in [("train", train), ("valid", valid), ("test", test)]:
        X = part.drop(columns=[target]).astype(float)
        y = part[target].astype(int)
        splits[name] = (X, y)
        print(f"{name:5s}: {len(part):5d} rows | positive rate = {y.mean():.2%} | positives = {int(y.sum())}")
    return splits

splits = temporal_split(df)
X_train, y_train = splits["train"]
X_valid, y_valid = splits["valid"]
X_test, y_test = splits["test"]
X_train_valid = pd.concat([X_train, X_valid], axis=0)
y_train_valid = pd.concat([y_train, y_valid], axis=0)
print(f"train+valid: {len(X_train_valid):5d} rows | positive rate = {y_train_valid.mean():.2%} | positives = {int(y_train_valid.sum())}")


train: 28823 rows | positive rate = 5.58% | positives = 1607
valid:  4117 rows | positive rate = 11.97% | positives = 493
test :  8236 rows | positive rate = 30.83% | positives = 2539
train+valid: 32940 rows | positive rate = 6.38% | positives = 2100


In [4]:
def choose_threshold(y_true, y_proba, thresholds=np.arange(0.05, 0.96, 0.01)):
    rows = []
    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)
        rows.append(
            {
                "threshold": float(round(threshold, 2)),
                "precision": precision_score(y_true, y_pred, zero_division=0),
                "recall": recall_score(y_true, y_pred, zero_division=0),
                "f1": f1_score(y_true, y_pred, zero_division=0),
                "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            }
        )
    table = pd.DataFrame(rows)
    best = table.sort_values(["f1", "recall", "precision"], ascending=False).iloc[0]
    return float(best["threshold"]), table


def metric_row(y_true, y_proba, threshold, scenario, split_name):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "scenario": scenario,
        "split": split_name,
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }


def build_lr_pipeline():
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
            ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
        ]
    )


In [5]:
scenarios = {
    "Scenario 1 - With duration (benchmark)": list(X_train.columns),
    "Scenario 2 - Without duration (realistic)": [col for col in X_train.columns if col != "duration"],
}

summary_rows = []
trained = {}
threshold_tables = {}

for scenario_name, feature_cols in scenarios.items():
    validation_model = build_lr_pipeline()
    validation_model.fit(X_train[feature_cols], y_train)

    valid_proba = validation_model.predict_proba(X_valid[feature_cols])[:, 1]
    best_threshold, threshold_table = choose_threshold(y_valid, valid_proba)

    # Final fit uses all past data available before the untouched test period.
    final_model = build_lr_pipeline()
    final_model.fit(X_train_valid[feature_cols], y_train_valid)
    test_proba = final_model.predict_proba(X_test[feature_cols])[:, 1]

    summary_rows.append(metric_row(y_valid, valid_proba, 0.50, scenario_name, "validation_at_0.50"))
    summary_rows.append(metric_row(y_valid, valid_proba, best_threshold, scenario_name, "validation_tuned"))
    summary_rows.append(metric_row(y_test, test_proba, 0.50, scenario_name, "test_at_0.50_after_refit"))
    summary_rows.append(metric_row(y_test, test_proba, best_threshold, scenario_name, "test_with_validation_threshold_after_refit"))

    trained[scenario_name] = {"pipeline": final_model, "features": feature_cols, "threshold": best_threshold}
    threshold_tables[scenario_name] = threshold_table

summary_df = pd.DataFrame(summary_rows)
display(summary_df.round(4))


,scenario,split,threshold,roc_auc,pr_auc,precision,recall,f1,balanced_accuracy
0,Scenario 1 - With duration (benchmark),validation_at_0.50,0.50,0.8682,0.4458,0.3217,0.8073,0.4601,0.7879
1,Scenario 1 - With duration (benchmark),validation_tuned,0.64,0.8682,0.4458,0.3743,0.7160,0.4916,0.7766
2,Scenario 1 - With duration (benchmark),test_at_0.50_after_refit,0.50,0.7171,0.5292,0.5720,0.4474,0.5021,0.6491
3,Scenario 1 - With duration (benchmark),test_with_validation_threshold_after_refit,0.64,0.7171,0.5292,0.6076,0.3726,0.4619,0.6327
4,Scenario 2 - Without duration (realistic),validation_at_0.50,0.50,0.6561,0.2011,0.1362,0.8682,0.2354,0.5595
5,Scenario 2 - Without duration (realistic),validation_tuned,0.70,0.6561,0.2011,0.2067,0.5030,0.2930,0.6202
6,Scenario 2 - Without duration (realistic),test_at_0.50_after_refit,0.50,0.6504,0.4614,0.3940,0.6806,0.4991,0.6070
7,Scenario 2 - Without duration (realistic),test_with_validation_threshold_after_refit,0.70,0.6504,0.4614,0.5084,0.4391,0.4713,0.6250


In [6]:
for scenario_name, info in trained.items():
    feature_cols = info["features"]
    threshold = info["threshold"]
    test_proba = info["pipeline"].predict_proba(X_test[feature_cols])[:, 1]
    test_pred = (test_proba >= threshold).astype(int)

    print("=" * 80)
    print(scenario_name)
    print(f"Validation-selected threshold: {threshold:.2f}")
    print(classification_report(y_test, test_pred, target_names=["No", "Yes"], zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, test_pred))


Scenario 1 - With duration (benchmark)
Validation-selected threshold: 0.64
              precision    recall  f1-score   support

          No       0.76      0.89      0.82      5697
         Yes       0.61      0.37      0.46      2539

    accuracy                           0.73      8236
   macro avg       0.68      0.63      0.64      8236
weighted avg       0.71      0.73      0.71      8236

Confusion matrix:
[[5086  611]
 [1593  946]]
Scenario 2 - Without duration (realistic)
Validation-selected threshold: 0.70
              precision    recall  f1-score   support

          No       0.76      0.81      0.79      5697
         Yes       0.51      0.44      0.47      2539

    accuracy                           0.70      8236
   macro avg       0.64      0.62      0.63      8236
weighted avg       0.69      0.70      0.69      8236

Confusion matrix:
[[4619 1078]
 [1424 1115]]


In [7]:
# Coefficients are in scaled-feature log-odds space.
for scenario_name, info in trained.items():
    model = info["pipeline"].named_steps["model"]
    coef_df = pd.DataFrame(
        {
            "feature": info["features"],
            "coefficient": model.coef_[0],
            "abs_coefficient": np.abs(model.coef_[0]),
        }
    ).sort_values("abs_coefficient", ascending=False)

    print("\n", scenario_name)
    display(coef_df.head(15).round(4))



 Scenario 1 - With duration (benchmark)


,feature,coefficient,abs_coefficient
8,euribor3m,4.8203,4.8203
1,duration,2.3702,2.3702
9,nr.employed,-2.0228,2.0228
5,emp.var.rate,-1.8966,1.8966
7,cons.conf.idx,-1.4788,1.4788
6,cons.price.idx,-1.0886,1.0886
45,month_nov,-1.0096,1.0096
44,month_may,-0.8032,0.8032
41,month_jul,-0.6449,0.6449
55,age_group_middle,-0.3612,0.3612



 Scenario 2 - Without duration (realistic)


,feature,coefficient,abs_coefficient
7,euribor3m,2.2491,2.2491
8,nr.employed,-0.9985,0.9985
4,emp.var.rate,-0.8474,0.8474
6,cons.conf.idx,-0.6082,0.6082
44,month_nov,-0.5718,0.5718
43,month_may,-0.4098,0.4098
40,month_jul,-0.3603,0.3603
37,contact_telephone,-0.3113,0.3113
5,cons.price.idx,-0.3070,0.3070
3,previous,-0.1734,0.1734


## Interpretation

Scenario 2 is the realistic one for pre-call targeting because `duration` is not available before a call happens. Scenario 1 is retained only as a leakage-aware benchmark.
